# **Klasszifikáló Klón**

<div style="font-size: 14px; color: #6e8192; line-height: 1.5;">
  <div style="display: flex; align-items: center; gap: 5px; margin-bottom: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🎯</span>
    <span>MI Országos Diákolimpia Válogató</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🧠</span>
    <span>Machine Learning</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🏆</span>
    <span>100 pont</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🗓️</span>
    <span>2025. Május 24.</span>
  </div>
</div>

**Név:** [ÍRD IDE A NEVED]

**Versenyzői Azonosító:** [ÍRD IDE A VERSENYZŐI AZONOSÍTÓD]

<img src="https://drive.google.com/uc?export=view&id=1Gs-32x6jJy1mbQV3j6554gSdwF5r_z3d" alt="klon" style="width:150px;">

## **Feladatleírás**:

Adott egy előre betanított neurális háló, amelyet klasszifikációs feladatra tanítottunk be egy meghatározott *training* adathalmazon. A feladat megoldásához a rendelkezésetekre áll:

* A betanítótt neurális háló
* A tanító adathalmaz és címkék
* A teszt adathalmaz egy kis részhalmaza (címkék nélkül)

A cél egy olyan gépi tanulási modell elkészítése, amely **nem neurális háló alapú**. Ennek a modellnek az a feladata, hogy **a teljes teszt adathalmazon a lehető legjobban közelítse meg a betanított neurális háló kimenetét**, azaz próbálja meg utánozni annak viselkedését.

**Elvárások a megoldással kapcsolatban:**

* Olyan gépi tanulási algoritmust válasszatok, amely nem neurális háló alapú.
* A végleges modell a teljes teszt adathalmazon lesz kiértékelve. A pontozás az accuracy metrika szerint, a következő skála alapján történik (az intervallum jobb oldala nyílt):
  * 0 - 0.5: 0 pont
  * 0.5 - 0.62: 20 pont
  * 0.62 - 0.66: 30 pont
  * 0.66 - 0.68: 40 pont
  * 0.68 - 0.70: 50 pont
  * 0.70 - 0.71: 70 pont
  * 0.71 - 0.728: 80 pont
  * 0.728 - 0.80: 90 pont
  * 0.80 - 1: 100 pont
* A pontozás során az első cellában beállított seed lesz használva. Ezen felül javasolt a modellek számára is random állapotot beállítani, mivel ez befolyásolhatja hogy milyen eredményt érnek el. A kiértékelés során minden kód csupán egyszer lesz futtatva.

## **Hasznos Linkek**

- [Scikit-Learn - Dokumentáció](https://scikit-learn.org/stable/)  

- [Catboost - Dokumentáció](https://catboost.ai/docs/en/concepts/tutorials)  

- [LightGBM - Dokumentáció](https://lightgbm.readthedocs.io/en/stable/)  

- [XGBoost - Dokumentáció](https://xgboost.readthedocs.io/en/release_3.0.0/)  


## **Szükséges Importok**

In [ ]:
import torch
import random
import numpy as np
import os

seed = 42

os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## **Adathalmaz és a Modell Letöltése**

In [ ]:
!gdown 1Ib-kv_832voOMhh0wnjTJ-z-9Q9B-t6E # Titkos modell

!gdown 1EG6QY3s8X6KKWtVdsswfeOfW3uAzOXU9 # Adatok

!gdown 1zf9FUPxVBDrafQUsIirkdZ_rb4a87o3b # Train-Test Split (ezen kell kiértékelni)

## **Erőforrások betöltése**

In [ ]:
model = torch.jit.load('secret_model.pt')
X_train, X_test_small, y_train, y_test_small = torch.load("data.pt")

---
## **Példa megoldás kezdete**
---

In [ ]:
# For evaluation
_, X_test, _, y_test = torch.load("train_test_split.pt")

In [ ]:
def generate_random_data(X_train, num_samples=1000):
    feature_num = X_train.size(1)

    random_data = torch.zeros(num_samples, feature_num)
    for i in range(feature_num):
        data = X_train[:, i]
        min_val = data.min().item()
        max_val = data.max().item()

        random_data[:, i] = min_val + (max_val - min_val) * torch.rand(num_samples)

    return random_data

random_data = generate_random_data(X_train, num_samples=100_000)

In [ ]:
X_train_combined = torch.cat((X_train, random_data), dim=0)
X_train_combined.size()

# X_train_combined = X_train  # Don't use generated data

torch.Size([101279, 11])

In [ ]:
# Get Neural Network labels

with torch.no_grad():
    model.eval()

    nn_test_preds = model(X_test)
    nn_train_preds = model(X_train_combined)

    _, nn_test_labels = torch.max(nn_test_preds, 1)
    _, nn_train_labels = torch.max(nn_train_preds, 1)


**Decision tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(max_depth=None, min_samples_split=2, random_state=42)
tree.fit(X_train_combined, nn_train_labels)

DecisionTreeClassifier(random_state=42)

In [ ]:
# Final score
tree_test_labels = tree.predict(X_test)
accuracy_score(nn_test_labels, tree_test_labels)

0.6375

**CatBoost**

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# Create and train the CatBoostClassifier
cat_model = CatBoostClassifier(
    iterations=500, depth=5, learning_rate=0.15,
   loss_function='MultiClass', verbose=False)

cat_model.fit(X_train_combined.numpy(), nn_train_labels.numpy())

In [ ]:
# Final score
cat_test_labels = cat_model.predict(X_test.numpy())
accuracy_score(nn_test_labels, cat_test_labels.flatten())

0.65

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create and train the Random Forest classifier
forest = RandomForestClassifier(
    n_estimators=100,         # number of trees in the forest
    max_depth=None,           # allow trees to grow until all leaves are pure or contain less than min_samples_split samples
    min_samples_split=4,      # minimum number of samples required to split an internal node
    random_state=42,
    n_jobs=1
)

forest.fit(X_train_combined, nn_train_labels)

RandomForestClassifier(min_samples_split=4, n_jobs=1, random_state=42)

In [ ]:
# Final score
forest_test_labels = forest.predict(X_test)
accuracy_score(nn_test_labels, forest_test_labels)

0.73125

---
## **Példa megoldás vége**
---

---

## 🎉 Gratulálunk!

A feladatsor végére értél - kiváló munkát végeztél!  

---